# Principal Component Analysis (PCA)

In previous labs, we explored unsupervised learning using $K$-means clustering. We often want to visualize our data to see if the clusters we found make sense. However, when we have many features (like the four physical measurements of the penguins), it's impossible to visualize the data in its original high-dimensional space.

Principal Component Analysis (PCA) is a technique for **dimensionality reduction**. it allows us to project high-dimensional data down to a lower-dimensional space (like 2D or 3D) while preserving as much of the variation in the data as possible.

## Reading in the Data

We will once again use the penguins data set.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

data_dir = "https://dlsun.github.io/stats112/data/"
df_penguins = pd.read_csv(data_dir + "penguins.csv").dropna().reset_index()
df_penguins['species']

For this analysis, we will focus on the four quantitative measurements: bill length, bill depth, flipper length, and body mass. We'll drop any rows with missing values.

In [ ]:
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
df_subset = df_penguins[features + ["species"]]

X = df_subset[features]
y = df_subset["species"]

## Many 2D views of 4 features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, (x, y) in zip(axes.ravel(), combinations(X.columns, 2)):
    df_subset.plot.scatter(x=x, y=y, c='species', ax=ax)

## Restrict to chinstrap 

In [ ]:
chinstrap = df_subset[df_subset['species'] == 'Chinstrap']
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, (x, y) in zip(axes.ravel(), combinations(X.columns, 2)):
    chinstrap.plot.scatter(x=x, y=y, c='teal', ax=ax)

## Dimensionality Reduction with PCA

Let's look at PCA of just `bill_length_mm` and `bill_depth_mm` first only within `Chinstrap`

In [ ]:
from sklearn.decomposition import PCA
X = chinstrap[features]
# Create a pipeline that scales the data and then applies PCA
pca_pipeline = make_pipeline(
    StandardScaler(),  # usually want to center and scale first
    PCA(n_components=2)
)

In [ ]:
# Fit and transform the data
X_pca = pd.DataFrame(pca_pipeline.fit_transform(X[['bill_length_mm', 'bill_depth_mm']]),
                     columns=['Principal Component #1', 'Principal Component #2'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
X.plot.scatter(x='bill_length_mm', y='bill_depth_mm', ax=axes[0])
X_pca.plot.scatter(x='Principal Component #1', y='Principal Component #2', ax=axes[1])

### `white=True`

In [ ]:
X_white = pd.DataFrame(
    make_pipeline(
        StandardScaler(),
        PCA(n_components=2, whiten=True)).fit_transform(chinstrap[['bill_length_mm', 'bill_depth_mm']]),
    columns=['Principal Component #1', 'Principal Component #2'])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
X.plot.scatter(x='bill_length_mm', y='bill_depth_mm', ax=axes[0])
X_white.plot.scatter(x='Principal Component #1', y='Principal Component #2', ax=axes[1])

Let's compare the two options.

In [ ]:
X_pca.cov()

The `whiten=True` just scales the scores by the half-lengths of the axes
of the standardized data.

In [ ]:
X_white.cov()

## Let's try with the full data

Now we have 4-dimensional `Chinstrap` data, and we've assigned each penguin to one of 3 clusters. How can we visualize this? We can use PCA to project the 4 features down to 2 dimensions.

In [ ]:
from sklearn.decomposition import PCA

# Create a pipeline that scales the data and then applies PCA
pca_pipeline = make_pipeline(
    StandardScaler(),
    PCA(n_components=4)
)

X = chinstrap[features]
y = chinstrap["species"]

# Fit and transform the data
X_pca = pca_pipeline.fit_transform(X)

In [ ]:
# Convert to a DataFrame for easier plotting
df_pca = pd.DataFrame(X_pca, columns=[f'Principal Component #{i}' for i in range(1, 5)])
df_pca.plot.scatter(x='Principal Component #1', y='Principal Component #2')

In [ ]:
fig, ax = plt.subplots()
explained_variance = pca_pipeline['pca'].explained_variance_ratio_
ax.plot(explained_variance)
ax.set_title('Scree plot')
ax.set_ylabel('Explained variance (as proportion)')
ax.set_xlabel('Component #')

### Cumulative version

In [ ]:
fig, ax = plt.subplots()
explained_variance = pca_pipeline['pca'].explained_variance_ratio_
ax.plot(explained_variance.cumsum())
ax.set_title('Percent variance explained')
ax.set_ylabel('Cumulative explained variance (as proportion)')
ax.set_xlabel('Component #')

## All the data

Let's cluster the penguins using all the features and visualize the labels
in the first 2 principal components.

In [ ]:
# Create a pipeline that scales the data and then applies K-Means
cluster = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=3, random_state=42, n_init="auto")
)

X = df_subset[features]

df_pca = pd.DataFrame(pca_pipeline.fit_transform(X), 
                     columns=[f'Principal Component #{i}' for i in range(1, 5)])
df_pca['clusters'] = cluster.fit(X).predict(X).astype(str)
df_pca['species'] = df_subset['species']

## Visualizing the Results

Now we can create two scatterplots of the first two principal components. 

First, let's color the points by the clusters identified by $K$-means. Then by species label

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for (ax, n, t) in zip(axes, 
                      ['clusters', 'species'],
                      ['K-Means Clusters', 'Species Labels']):
    df_pca.plot.scatter(x='Principal Component #1',
                        y='Principal Component #2',
                        c=n,
                        ax=ax)

    ax.set_xlabel("First Principal Component (PC1)")
    ax.set_ylabel("Second Principal Component (PC2)")
    ax.set_title(f"{t} in PCA Space")


## Exercise

Do a similar task on the Breast Tissue data:

In [ ]:
df = pd.read_csv("https://datasci112.stanford.edu/data/BreastTissue.csv").dropna().set_index('Case #')
